# 04 - Steering, neuron baseline, and UMAP

Owner: Areeb

Activation patching, SAE feature steering, neuron monosemanticity
baseline, and a UMAP of the SAE decoder.


## Setup

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import torch
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
MODEL_CFG_YAML = REPO / "configs" / "model_decoder_small.yaml"
MODEL_CHECKPOINT = REPO / "checkpoints" / "decoder_small_best_model.pt"
SAE_CHECKPOINT = REPO / "checkpoints" / "dev_sae.pt"
TOKENIZER_PATH = REPO / "tokenizer_output" / "tokenizer.json"

TARGET_LAYER = 2
SAE_HOOK_NAME = f"blocks.{TARGET_LAYER}.hook_resid_post"
MLP_HOOK_NAME = f"blocks.{TARGET_LAYER}.mlp.hook_post"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## Load model and SAE

In [ ]:
from mid.config import load_configs
from mid.model.hooked_model import load_checkpoint

model_cfg, _ = load_configs(str(MODEL_CFG_YAML))
model = load_checkpoint(str(MODEL_CHECKPOINT), model_cfg)
model.eval()
print(f"model: {sum(p.numel() for p in model.parameters()):,} params, "
      f"d_model={model_cfg.d_model}, n_layers={model_cfg.n_layers}")


In [ ]:
from tokenizers import Tokenizer as HFTokenizer
from transformers import PreTrainedTokenizerFast

hf_tok = HFTokenizer.from_file(str(TOKENIZER_PATH))
pt_tok = PreTrainedTokenizerFast(
    tokenizer_object=hf_tok,
    bos_token="<|bos|>", eos_token="<|eos|>",
    unk_token="<|unk|>", pad_token="<|pad|>",
)
# set_tokenizer() tries to fetch a BOS variant from the HF Hub, which fails
# for our local BPE tokenizer. Wire it on by hand and disable BOS prepending.
model.tokenizer = pt_tok
model.cfg.tokenizer_prepends_bos = False
model.cfg.default_prepend_bos = False
print(model.to_tokens("HAMLET:").shape)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class SAECfg:
    hook_name: str
    d_in: int
    d_sae: int

class DevSAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_enc = nn.Parameter(torch.randn(cfg.d_in, cfg.d_sae) * 0.01)
        self.b_enc = nn.Parameter(torch.zeros(cfg.d_sae))
        self.W_dec = nn.Parameter(torch.randn(cfg.d_sae, cfg.d_in) * 0.01)
        self.b_dec = nn.Parameter(torch.zeros(cfg.d_in))
    def encode(self, x): return F.relu((x - self.b_dec) @ self.W_enc + self.b_enc)
    def decode(self, z): return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z = self.encode(x); return self.decode(z), z

# dev SAE placeholder; replace with SAELens load when the real SAE is ready
_blob = torch.load(SAE_CHECKPOINT, map_location=DEVICE)
sae = DevSAE(SAECfg(**_blob["cfg"])).to(DEVICE)
sae.load_state_dict(_blob["state_dict"])
sae.eval()
print(f"SAE: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}, hook={sae.cfg.hook_name}")


## Activation patching

In [ ]:
from mid.analysis.patching import patch_activation, compare_outputs

clean_prompt     = "HAMLET:\nTo be, or not to be, that is the"
corrupted_prompt = "FALSTAFF:\nA plague upon thy house, thou villainous"

clean_len = model.to_tokens(clean_prompt).shape[1]
corr_len  = model.to_tokens(corrupted_prompt).shape[1]
pos = min(clean_len, corr_len) - 1
print(f"clean_len={clean_len}, corr_len={corr_len}, patching at pos={pos}")

for layer in range(model_cfg.n_layers):
    hook = f"blocks.{layer}.hook_resid_post"
    delta = patch_activation(model, clean_prompt, corrupted_prompt, hook, pos)
    print(f"layer {layer}: patched_loss - clean_loss = {delta:+.4f}")


In [ ]:
from functools import partial

_, corrupted_cache = model.run_with_cache(model.to_tokens(corrupted_prompt))
layer_to_show = 2
hook_to_show = f"blocks.{layer_to_show}.hook_resid_post"

def patch_hook(act, hook, src):
    act[:, pos, :] = src[:, pos, :]
    return act

top = compare_outputs(
    model, clean_prompt, hook_to_show,
    partial(patch_hook, src=corrupted_cache[hook_to_show]),
    top_k=10,
)
print("CLEAN top-10:", top["clean"])
print("PATCHED top-10:", top["patched"])


## Feature steering

In [ ]:
from mid.analysis.patching import steer_with_feature

FEATURE_IDX = 42
PROMPT = "HAMLET:\nTo be"

for coeff in [0.0, 2.0, 5.0, -2.0]:
    out = steer_with_feature(
        model, sae, FEATURE_IDX, coefficient=coeff,
        prompt=PROMPT, max_new_tokens=30,
    )
    print(f"coeff={coeff:+.1f}: {out}\n")


## Neuron baseline

In [ ]:
from mid.analysis.neuron_baseline import (
    top_activating_neurons, score_monosemanticity, summarize_neuron,
)
import numpy as np

val_ids = np.load(REPO / "tokenizer_output" / "val.npy")
seq_len = 128
n_seqs = 64
slab = val_ids[: n_seqs * seq_len].reshape(n_seqs, seq_len)
dataset_tokens = torch.from_numpy(slab).long().to(DEVICE)
print("dataset_tokens:", dataset_tokens.shape)


In [ ]:
neuron_top = top_activating_neurons(
    model, dataset_tokens, MLP_HOOK_NAME, k=10, context_window=6,
)
print(f"scored {len(neuron_top)} neurons")

neuron_scores = score_monosemanticity(neuron_top)
example = summarize_neuron(0, neuron_top)
print("neuron 0 top tokens:", example["top_tokens"][:5])
print("neuron 0 sample context:", example["contexts"][0][0][:120])


### LLM-scored subset

Run the Claude-based scorer on a small sample. Gated on `ANTHROPIC_API_KEY`
so the notebook still runs without credentials.


In [ ]:
import os
if os.environ.get("ANTHROPIC_API_KEY"):
    sample_top = {i: neuron_top[i] for i in list(neuron_top)[:10]}
    llm_scores = score_monosemanticity(sample_top, llm_client="anthropic")
    for i, s in sorted(llm_scores.items(), key=lambda kv: -kv[1]):
        print(f"neuron {i}: llm={s:.2f}  heuristic={neuron_scores[i]:.2f}")
else:
    print("ANTHROPIC_API_KEY not set; skipping LLM scoring demo")


### SAE feature baseline

In [ ]:
with torch.no_grad():
    _, resid_cache = model.run_with_cache(dataset_tokens, names_filter=SAE_HOOK_NAME)
    resid = resid_cache[SAE_HOOK_NAME]
    feat = sae.encode(resid)

def _top_contexts_from_acts(acts, tokens, k=10, ctx=6):
    B, S, F_ = acts.shape
    flat = acts.reshape(-1, F_)
    top_vals, top_pos = flat.topk(k, dim=0)
    out = {}
    for fi in range(F_):
        rows = []
        for r in range(k):
            idx = top_pos[r, fi].item()
            b, s = divmod(idx, S)
            a, z = max(0, s - ctx), min(S, s + ctx + 1)
            snippet = model.to_string(tokens[b, a:z])
            rows.append((snippet, top_vals[r, fi].item()))
        out[fi] = rows
    return out

feat_top = _top_contexts_from_acts(feat, dataset_tokens, k=10, ctx=6)
feature_scores = score_monosemanticity(feat_top)
print(f"scored {len(feature_scores)} SAE features")


### Side-by-side

In [ ]:
from mid.analysis.neuron_baseline import compare_to_sae

summary = compare_to_sae(neuron_scores, feature_scores, threshold=0.5)
for k, v in summary.items():
    if k.endswith("_sorted"):
        continue
    print(f"{k:>30s}: {v}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(summary["neuron_scores_sorted"], bins=30, alpha=0.5, label="neurons")
ax.hist(summary["feature_scores_sorted"], bins=30, alpha=0.5, label="SAE features")
ax.set_xlabel("monosemanticity score")
ax.set_ylabel("count")
ax.set_title("Monosemanticity: raw neurons vs SAE features")
ax.legend()
plt.show()


## UMAP of SAE decoder directions

In [ ]:
from mid.analysis.umap_features import project_features, plot_feature_map

emb, fidx = project_features(sae, n_components=2, n_neighbors=15,
                             min_dist=0.1, metric="cosine")
print("embeddings:", emb.shape)

fig = plot_feature_map(
    emb,
    highlight_idxs=[FEATURE_IDX],
    title=f"SAE features (layer {TARGET_LAYER} residual stream)",
)
plt.show()


### LLM vs heuristic + save scores

In [ ]:
import json
from pathlib import Path

scores_out = {
    "neuron_heuristic": neuron_scores,
    "feature_heuristic": feature_scores,
}

if os.environ.get("ANTHROPIC_API_KEY"):
    feat_sample = {i: feat_top[i] for i in list(feat_top)[:10]}
    feature_llm_scores = score_monosemanticity(feat_sample, llm_client="anthropic")
    scores_out["neuron_llm"] = llm_scores
    scores_out["feature_llm"] = feature_llm_scores

    fig, ax = plt.subplots(figsize=(6, 6))
    n_x = [neuron_scores[i] for i in llm_scores]
    n_y = [llm_scores[i] for i in llm_scores]
    f_x = [feature_scores[i] for i in feature_llm_scores]
    f_y = [feature_llm_scores[i] for i in feature_llm_scores]
    ax.scatter(n_x, n_y, label="neurons", alpha=0.7)
    ax.scatter(f_x, f_y, label="SAE features", alpha=0.7, marker="^")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1)
    ax.set_xlabel("heuristic score")
    ax.set_ylabel("LLM score")
    ax.set_title("LLM vs heuristic monosemanticity (sampled subset)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(); ax.grid(alpha=0.2)
    plt.show()
else:
    print("ANTHROPIC_API_KEY not set; saving heuristic scores only")

out_path = REPO / "data" / "monosemanticity_scores.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps({
    k: {str(i): float(v) for i, v in d.items()} for k, d in scores_out.items()
}, indent=2))
print(f"wrote {out_path.relative_to(REPO)}")
